# Compare Detected Storm 
In this notebook, we are going to compare the monthly detected storm using the function developed by Prayla and compare it to the detected storm on Lobeto et al., 2024 paper. However, we don't have the exact coordinate of the previous study key locations (in total 24 locations). Therefore, the location described on the rest of the notebook is approximately in the same area as the study's paper, but not exactly the same. It is also adjusted according to the availability grid of ERA5. 

In [ ]:
# load the necessary libraries 

import cartopy.crs as ccrs
import geopandas as gpd
import hvplot.pandas 
import hvplot.xarray
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 
import panel as pn
import scipy.io
import scipy.signal as signal
import xarray as xr

from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from datetime import datetime, timedelta
from pcr import storm
from shapely.geometry import Point


## ERA5 Dataset 
A set of [ERA5 hourly time-series data on single levels from 1940 to present](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=download) is prepared for this notebook. This dataset is a subset if some parametersof full CDS ERA5 dataset on 0.25 degree resolution and stored in *Analysis Ready Cloud Optimised (ARCO)* format. In which, the retrieving long time-series for a single point is far more efficient. 

> **_NOTE:_** The generation of this dataset may subject to change over time. It is still regarded as "experimental" product.  

Also, I tried multiple of ways to access ERA5 data: 
1. through `cdsapi` from original dataset 
2. through earth_data
3. through `cdsapi` from time-series dataset (the one showed in this notebook)

In [ ]:
path_to_data = '../data/ERA5/ts/unzipped/'

# create a list to store station data 
station_data = []

# loop through all .nc files
for i in range(24):

    ds = xr.open_dataset(f'{path_to_data}p{i+1}_ts.nc')
    
    lon = ds.longitude.values.item()
    lat = ds.latitude.values.item()

    geom = Point(lon, lat)

    # append to the list
    station_data.append({
        'station': f'p{i+1}',
        'longitude': lon,
        'latitude': lat,
        'geometry': geom
    })

# create a GeoDataFrame form the list 
gdf_era5 = gpd.GeoDataFrame(station_data, geometry='geometry',crs='EPSG:4326')

Show the location of each point. The point is not exactly the same as Lobeto, because we don't have that yet. 

In [ ]:
# read the point of interest from json 
poi_gdf = gpd.read_file('../data/point_of_interest.json')

# create figure with PlateCarree projection 
figpoi, ax = plt.subplots(1, 1, subplot_kw=dict(projection=ccrs.PlateCarree()), figsize=(10, 6))

# set up map background
ax.stock_img()

# set the title
ax.set_title("Area of Interest")

# Plot POIs on the map 
poi_gdf.plot(ax=ax, marker='o', alpha=1, color='red')

# Annotate each point with the station label with offset (3,3) in points, xy value
for x, y, label in zip(poi_gdf.geometry.x, poi_gdf.geometry.y, poi_gdf.station):
    ax.annotate(label, xy=(x, y), xytext=(-10, -12), textcoords='offset points')

# compare the predefined POI location with the ERA5 available points 
gdf_era5.plot(ax=ax, marker='x', color='blue', markersize=10, label='ERA5 Location')

# Add a legend 
ax.legend()

# Add the longitude and latitude ticks
ax.set_xticks(range(-180, 181, 60))  # Longitude ticks every 60 degrees
ax.set_yticks(range(-90, 91, 20))  # Latitude ticks every 20 degrees

# Use formatters to display longitude and latitude in a readable format
ax.xaxis.set_major_formatter(LongitudeFormatter())
ax.yaxis.set_major_formatter(LatitudeFormatter())

# Add gridlines with specific intervals
ax.grid(True, linestyle='--', color='gray', alpha=0.5)

# Show the plot
plt.show()


The time-series consist of hourly 40-years data from 1979 to 2020. Parameters that are included are *significant wave height*, *mean wave period*, *mean wave direction*. 

In [ ]:
# Create a list of file paths for the 24 datasets
file_paths = [f'../data/ERA5/ts/unzipped/p{i}_ts.nc' for i in range(1, 25)]

# Read in the datasets and extract the "swh" variable for each one
ds_list = [xr.open_dataset(path) for path in file_paths]

# Define a function to plot the line graph for a specific dataset
def plot_station(index):
    ds = ds_list[index]
    title = f'Station P{index+1} Location: lat = {ds.latitude.values.item():.02f}\N{degree sign}, lon = {ds.longitude.values.item():.02f}\N{degree sign}'
    return ds['swh'].hvplot.line(width=600, grid=True, title=title)

# Create a dropdown widget for selecting the station (file)
station_labels = [f'P{i}' for i in range(1, 25)] # list of station name "P1, "P2", ..., "P24"
station_dropdown = pn.widgets.Select(name='Select Station', options=station_labels, value='P1') # dropdown options with default value 'P1'
# station_dropdown = pn.widgets.Select(name='Select Station', options=list(range(1, 25)), value=0)

# change the dropdown value to index (0-23)
def update_plot(station_label): 
    index = int(station_label[1:]) - 1  # Extract index from label (e.g., 'P1' -> 0)
    return plot_station(index)

# Create a dynamic plot that updates based on the dropdown selection, binding the "station_label" parameter as the dropdown 
dynamic_plot = pn.bind(update_plot, station_label=station_dropdown)

# Layout the dropdown and the plot together
layout = pn.Column(station_dropdown, dynamic_plot)

# Display the layout
layout.servable()


## Detect Storm
Detect the storm using `storm.detect` function. 

In [ ]:
# intialize stattic variables for detect storm 
ts_hs = 95
ts_dur = 12

# initialize dataframes to store results. Rows of 
monthly_perc = pd.DataFrame(
    index=np.arange(1, 13, 1)
)

monthly_count = pd.DataFrame(
    index=np.arange(1, 13, 1)
)

# loop through all .nc files
for i in range(24):

    ds = xr.open_dataset(f'{path_to_data}p{i+1}_ts.nc')

    hs = ds['swh'].values
    dir = ds['mwd'].values
    tp = ds['mwp'].values

    data_start = ds.valid_time.values[0]
    time = (ds.valid_time.values - data_start).astype('timedelta64[h]').astype(float) / 24

    try:
        storm_df, _ = storm.detect(hs, dir, tp, time, ts_hs, ts_dur)

        hours = storm_df['start'].values * 24

        storm_df['months'] = [(pd.to_datetime(data_start) + timedelta(hours=hour)).month for hour in hours]

        monthly_count[f'P{i+1}'] = storm_df[['hs_max', 'months']].groupby('months').count()#.rename(columns={
        #     'hs_max': row.station
        # })

        monthly_perc[f'P{i+1}'] = monthly_count[f'P{i+1}'] / monthly_count[f'P{i+1}'].sum() * 100
        
    except IndexError as e:
        print(f"IndexError occurred at station {i+1}, breaking the loop.")
        pass  # Exit the loop if an IndexError is raised

Visualize the result using a histogram of monthly storm percentage. We can also compare it to the figure 2 of Lobeto et al., 2024

In [ ]:
# import data from hector

data_fig3 = scipy.io.loadmat('../data/output/data_Figure3.mat')

nsws = data_fig3['Nsws']
nws = data_fig3['Nws']

In [ ]:
# create subplots for 24 locations (6 columns and 4 rows)

n_col = 6
n_row = 4

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
fig, axs = plt.subplots(ncols=n_col, nrows=n_row, figsize=(25,12))#, layout='constrained')
width = 0.4
x = np.arange(12)

station = monthly_perc.columns.values

# iter rows
for i in range(n_row):
    # iter cols
    for j in range(n_col):
        idx = j + i * n_col
        if idx < len(station):
            axs[i,j].grid(linestyle='--', alpha=0.3, zorder=0)
            axs[i,j].bar(x-width/2, monthly_perc[station[idx]], width, label= "PCR", zorder=3)
            axs[i,j].bar(x+width/2, nws[:, int(station[idx][1:])-1], width, label="Lobeto et al., 2024", zorder=3)
            axs[i,j].set_title(station[idx])
            axs[i,j].set_ylim([0, 60])
            axs[i,j].set_xticks(x, months)
            axs[i,j].set_ylabel('Percentage Frequency of Occurence')
        else:
            axs[i,j].axis('off')

# create legend on the bottom of the figure. bbox_to_anchor=point refer to the 'loc' position. frameon=False to remove box around legend
fig.legend(['PCR', 'Lobeto et al., 2024'], loc='center', ncols=2, bbox_to_anchor=(0.5, -0.02), frameon=False) 

fig.tight_layout()
fig.suptitle('Monthly percentage frequency of occurrence of wave storm events at key locations', y=1.02)

plt.show()

In [ ]:
# create a dataframe for Lobeto et al., 2024 data
lobeto = pd.DataFrame(nws, columns=[f'P{i}' for i in range(1, 25)], index=np.arange(1, 13, 1))


# create dropdown to select station
station_labels = [f'P{i}' for i in range(1, 25)] 
station_dropdown = pn.widgets.Select(name='Select Station', options=station_labels, value='P1')

# function to generate chart
def bar_comparison(station_label): 
    combined_df = pd.DataFrame({
        'PCR': monthly_perc[station_label],
        'Lobeto': lobeto[station_label]
    })

    combined_df['month'] = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


    grouped_fig = combined_df.hvplot.bar(
        x='month', 
        y=['PCR', 'Lobeto'],
        title=f'Station {station_label}',
        group_label='source', 
        width=600,
        yformatter='%i',
    )

    return grouped_fig.opts(multi_level=False)

# bind the dropdown to the function
dynamic_plot = pn.bind(bar_comparison, station_label=station_dropdown)

# layout the dropdown and the plot together
layout = pn.Column(station_dropdown, dynamic_plot)

# display the layout
layout.servable()

There are differences on P6 and P13, which is lies on the North Pacific Ocean, on the contrasting sides (closer to Central America on eastern side and on the South China Sea on the western side). There are also a little differences on P5 which also located in North Pacific Ocean, just off coast of Japan. 

In [ ]:
# examine storm in P6
ds_p6 = xr.open_dataset(f'{path_to_data}p6_ts.nc')

hs_p6 = ds_p6['swh'].values
tp_p6 = ds_p6['mwp'].values
dir_p6 = ds_p6['mwd'].values

storm_df_p6, storm_ts_p6 = storm.detect(hs_p6, dir_p6, tp_p6, time, ts_hs, ts_dur)

storm_ts_p6['date'] = data_start + [np.timedelta64(int(hour*24), 'h') for hour in storm_ts_p6['time']]

# first figure
full_ts = ds_p6['swh'].hvplot.line(
    width=600, 
    grid=True
)

# second figure
storm_mark = storm_ts_p6.hvplot.scatter(
    x='date', 
    y='hs',
    color='red', 
)

full_ts * storm_mark

P6 is located in Central America off the west coast of America. The storm persist throughout the year. 

In [ ]:
# examine storm in P3
ds_p3 = xr.open_dataset(f'{path_to_data}p3_ts.nc')

hs_p3 = ds_p3['swh'].values
tp_p3 = ds_p3['mwp'].values
dir_p3 = ds_p3['mwd'].values

storm_df_p3, storm_ts_p3 = storm.detect(hs_p3, dir_p3, tp_p3, time, ts_hs, ts_dur)

storm_ts_p3['date'] = data_start + [np.timedelta64(int(hour*24), 'h') for hour in storm_ts_p3['time']]

# first figure
full_ts = ds_p3['swh'].hvplot.line(
    width=600, 
    grid=True
)

# second figure
storm_mark = storm_ts_p3.hvplot.scatter(
    x='date', 
    y='hs',
    color='red'
)

full_ts * storm_mark

P13 is located in South China Sea. The storm pattern here is following the Northern winter where the storm is mostly concentrated in month December, January, and February.

## Independent storm events 
there are some differences between Hector's workflow in detecting storm, which include a criteria to consider storm event independency. Two wave storm events are considered independent when the time between two consecutive Hs peaks over the threshold exceeds **48 hours**.  
In this case, the case on point 6 is used. 

In [ ]:
# set up the variable 
hs = ds_p6['swh'].values 
dir = ds_p6['mwd'].values
tp = ds_p6['mwp'].values

data_start = ds_p6.valid_time.values[0]
time = (ds_p6.valid_time.values - data_start).astype('timedelta64[h]').astype(float) / 24

# obtain index where hs exceeds threshold
threshold = np.percentile(hs, ts_hs)
storm_id = np.where(hs >= threshold, 1, 0)

# initiate storm id to track time 
delta_t = (time[1]-time[0])*24  # assuming uniform time steps in hour
storm_dur = storm_id * delta_t  # duration in hours

# get cumulative sum of time where hs exceeds threshold
storm_dur_cum = np.zeros_like(storm_dur) 
for i in range(1, len(storm_dur)):
    if storm_dur[i] > 0:
        storm_dur_cum[i] = storm_dur_cum[i-1] + storm_dur[i]

# find the peak of duration at the end of each storm
# end = peak (in storm.py)
ends, props = signal.find_peaks(storm_dur_cum, height=ts_dur)
mask = props["peak_heights"] > ts_dur
end_indices = ends[mask]

# find the start indices by subtracting end indices by the duration of storm 
storm_nr = (storm_dur_cum[end_indices] / delta_t).astype(int)
start_indices = end_indices - storm_nr

# find the max indices
max_indices = [hs[start:end].argmax() for start, end in zip(start_indices, end_indices+1)] + start_indices

# find the peaks which are assumed not independent
dep_storm_nr = np.where(np.diff(max_indices)*delta_t < 48)


peak_dep_indices = max_indices[dep_storm_nr]

# remove start on index dep_storm_nr + 1 
start_indices_new = np.array([start_indices[i] for i in range(len(start_indices)) if i not in dep_storm_nr[0] + 1])

# remove end on index dep_storm_nr
end_indices_new = np.array([end_indices[i] for i in range(len(start_indices)) if i not in dep_storm_nr[0]])

In [ ]:
# visualize each element of the storm 
hs_df = pd.DataFrame({
    'hs': hs, 
    'time': ds_p6['valid_time'].values
})

hs_plot = hs_df.hvplot.line(
    x='time', 
    y='hs'
) 

end_plot = hs_df.loc[end_indices].hvplot.scatter(
    x='time', 
    y='hs', 
    color='red'
)

start_plot = hs_df.loc[start_indices].hvplot.scatter(
    x='time', 
    y='hs',
    color='green'
)

max_plot = hs_df.loc[max_indices].hvplot.scatter(
    x='time', 
    y='hs',
    marker='x', 
    color='orange'
)

dep_plot = hs_df.loc[peak_dep_indices].hvplot.scatter(
    x='time', 
    y='hs',
    marker='x', 
    color='red'
)

end_plot_in = hs_df.loc[end_indices_new].hvplot.scatter(
    x='time', 
    y='hs', 
    color='green',
    marker='x'
)

start_plot_in = hs_df.loc[start_indices_new].hvplot.scatter(
    x='time', 
    y='hs',
    color='red',
    marker='x'
)


hs_plot * end_plot * start_plot * max_plot * dep_plot * end_plot_in * start_plot_in

In [ ]:
# implemented at storm.py
import importlib
importlib.reload(storm)

In [ ]:
# examine storm in P6
ds_p6 = xr.open_dataset(f'{path_to_data}p6_ts.nc')

hs_p6 = ds_p6['swh'].values
tp_p6 = ds_p6['mwp'].values
dir_p6 = ds_p6['mwd'].values
ts_between = 48

storm_df_p6, storm_ts_p6 = storm.detect(hs_p6, dir_p6, tp_p6, time, ts_hs, ts_dur, ts_between)

storm_ts_p6['date'] = data_start + [np.timedelta64(int(hour*24), 'h') for hour in storm_ts_p6['time']]

# first figure
full_ts = ds_p6['swh'].hvplot.line(
    width=600, 
    grid=True
)

# second figure
storm_mark = storm_ts_p6.hvplot.scatter(
    x='date', 
    y='hs',
    color='red', 
)

full_ts * storm_mark * end_plot * start_plot * end_plot_in * start_plot_in